# Lab 1.1. Reading files: CSV and columnar data

**Module 2. Session 1: ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

In this lab, you will learn how to:

1. Inspect a file before loading it.
2. Read CSV files with different separators, decimal conventions, encodings and date formats.
3. Check what one row of a dataset represents.
4. Identify and handle missing values encoded with sentinel values.
5. Compare CSV and Parquet in terms of size, speed and type preservation.

### How to work

Each exercise includes:

1. a short explanation or question
2. a code cell for you to complete
3. a check to verify your result

Exercises marked **Extension** are optional.

Before finishing, run the notebook from top to bottom and check that all cells execute without errors.

> The source data use Spanish column names such as `fecha`, `intensidad` and `vmed`. We will keep those names as they appear in the original files, while using English for the variables and columns we create.


## 0. Environment

Record the versions you worked with.

In [ ]:
import sys, platform
import pandas as pd, numpy as np, matplotlib
print("Python     ", sys.version.split()[0], "|", platform.system())
for m in (pd, np, matplotlib):
    print(f"{m.__name__:<11}", m.__version__)

## 0. Setup

Before starting, we need to locate the course folder and make sure the notebook can access the data files.

This setup cell is designed to work in different environments:

* on your own computer
* in Google Colab using the course repository
* in Google Colab using the shared Google Drive folder

Run this cell first.

If everything is working correctly, you should see a list of the files available in the course folder.

> **If the file list appears, the notebook has found the data correctly and you are ready to continue.**


In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")

    root = Path("/content/acs-mod2")                   # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root

    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

## Exercise 0. The project folder *(10 minutes)*

Real project data rarely comes in a single, clean format. More often, you receive a folder containing files created by different tools, for different purposes, and following different conventions.

Before opening these files with Python, it is useful to first understand what each file actually contains.

> **File extensions are useful clues, but they do not always guarantee the actual format of a file.**

In this exercise, we will use the `peek()` helper to inspect the first bytes of each file and make an initial assessment of its format.

You do not need to understand how `peek()` works yet. For now, focus on one important idea:

**Successfully decoding a file as UTF-8 does not necessarily mean that it is a text file.**

Binary files can also contain byte sequences that are valid UTF-8, including null bytes (`\x00`).


In [ ]:
def peek(path, n=64):
    """First bytes of a file and a provisional text/binary verdict."""
    b = Path(path).read_bytes()[:n]
    nulls = b.count(0)
    printable = sum(1 for c in b if 32 <= c < 127 or c in (9, 10, 13))
    verdict = "binary" if nulls > 0 or printable / max(len(b), 1) < 0.85 else "text"
    print(f"{Path(path).name:<26} {verdict:<8} {b[:44]!r}")

files = [
    "aforos_202509.csv", "pm_ubicaciones.csv", "imd_septiembre.xlsx",
    "intensidades_2024.xls", "municipios.geojson", "municipios_shp/municipios.shp",
    "municipios_shp/municipios.dbf", "municipios_shp/municipios.prj",
    "mdt25_madrid.tif", "PNOA_2020_0559.las", "estructura_rev07.ifc", "PZ07_20250915.dat",
]
for f in files:
    peek(DATA / f)

### Inspect the files

Work with your partner and, **without opening the files with any other program**, classify each one as either `text` or `binary`.

Then complete the second dictionary with any files whose extension does not match their actual format.

If you are not sure what a file is, write down **what you can observe** rather than guessing. In real projects, identifying what is known and what is uncertain is also a valid result.

Before running the check, make a prediction:

> **How many files do you think have a misleading extension?**


In [ ]:
diagnosis = {
    "aforos_202509.csv": "text",          # worked example
    "pm_ubicaciones.csv": "?",
    "imd_septiembre.xlsx": "?",
    "intensidades_2024.xls": "?",
    "municipios.geojson": "?",
    "municipios.shp": "?",
    "municipios.dbf": "?",
    "municipios.prj": "?",
    "mdt25_madrid.tif": "?",
    "PNOA_2020_0559.las": "?",
    "estructura_rev07.ifc": "?",
    "PZ07_20250915.dat": "?",
}
# YOUR CODE HERE
raise NotImplementedError

# Which files lie about their extension, and what are they really?
# Accepted values: "html", "csv", "delimited text", "json", "xml"
lying = {
    # "file_name": "what it really is",
}
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
key = {
    "aforos_202509.csv": "text", "pm_ubicaciones.csv": "text",
    "imd_septiembre.xlsx": "binary", "intensidades_2024.xls": "text",
    "municipios.geojson": "text", "municipios.shp": "binary",
    "municipios.dbf": "binary", "municipios.prj": "text",
    "mdt25_madrid.tif": "binary", "PNOA_2020_0559.las": "binary",
    "estructura_rev07.ifc": "text", "PZ07_20250915.dat": "text",
}
wrong = [k for k in key if diagnosis.get(k) != key[k]]
assert not wrong, f"Revisit text/binary for: {wrong}"
assert set(lying) == {"intensidades_2024.xls", "PZ07_20250915.dat"}, (
    "The two liars are an .xls and a .dat. Look at their first bytes again.")
assert lying["intensidades_2024.xls"] == "html", "A file that starts with <html> is not Excel"
assert lying["PZ07_20250915.dat"] in ("csv", "delimited text"), (
    "A .dat with quotes and commas is delimited text, with four header lines")
print("Checks passed: 12 files classified, 2 extensions unmasked.")

### Debrief

Three ideas to take away from this exercise:

1. **The file extension does not always tell you the real format.**
   `intensidades_2024.xls` is actually an HTML file. This is common in public data portals, where a web table is exported with an `.xls` extension so that Excel can open it.

   In this case, `read_excel()` fails, while `read_html()` works.

2. **You do not need to recognise every file format immediately.**
   `estructura_rev07.ifc` may be unfamiliar, but we can still observe that it is a text file and that it begins with:

   ```text
   ISO-10303-21
   ```

   That gives us enough information to identify the standard with a quick search.

   > A professional answer is not always “I know what this is”. Sometimes it is: **“I do not know yet, but I know what I can observe and how to find out.”**

3. **Some formats are made up of several files.**
   A shapefile, for example, is not a single file. It usually consists of several related files, and they are not all binary.

   The `.prj` file contains the coordinate reference system as **WKT text**, so it can be opened and read in a text editor.

   We will look at this in more detail in **Lab 1.2**.

---

The rest of the session will show that, despite the variety of file extensions, most of these files can be understood through a relatively small number of underlying data structures.


## 1. Before `pandas`: look at the raw content

A file extension gives us a clue about the format, but it does not guarantee what is actually inside the file.

For example, a file ending in `.csv` does not necessarily use commas as separators.

Before loading a file with `pandas`, it is often useful to inspect its raw content first.

In Python, `repr()` helps us see characters that are normally interpreted when printed, such as:

* `\n` → new line
* `\t` → tab

This makes it easier to understand how the file is structured before deciding how to read it.


In [ ]:
counts_path = DATA / "aforos_202509.csv"
print("Size:", round(counts_path.stat().st_size / 1e6, 2), "MB")

with open(counts_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 3:
            break

### What can we observe?

Before continuing, look carefully at the raw content and identify three things:

* the field separator is **not a comma**
* decimal values use a **comma**
* dates follow a **day/month/year** format

These details matter because `pandas` will not always infer them correctly.

> Before reading a file, it is useful to understand how its values are actually encoded and separated.


### Exercise 1. The naive read

Read the file using `pd.read_csv()` and only specify the encoding. Store the result in a variable called `naive`.

The code will run without raising an error, but that does not mean the data has been read correctly.

Before answering, inspect:

* the column names
* the **index**
* the first few rows

> The important point here is that some data-reading errors are **silent**: the code works, but the resulting structure is wrong.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("Columns detected:", naive.shape[1])
print("Column name:", list(naive.columns))
print("Index:", list(naive.index[:2]))
naive.head(3)

In [ ]:
assert naive.shape[1] == 1, f"Expected a single column, got {naive.shape[1]}"
assert "intensidad" not in naive.columns, "pandas has not split the fields"
assert not isinstance(naive.index, pd.RangeIndex), (
    "Part of the data should have ended up in the index")
print("Checks passed: the naive read does not split the fields")
print("and also pushes part of each row into the index.")

### What happened?

Complete the explanation:

- `pandas` detected **___ column(s)**.
- The default separator used by `read_csv()` is **___**.
- The actual separator in this file is **___**.
- The commas that appear in the data represent **___**.
- Some values therefore ended up in the **___** instead of in separate columns.

> The code ran without an error, but the resulting DataFrame is incorrect. This is an example of a **silent data-reading error**.


### Exercise 2. The correct read

Now read the file again, this time using the arguments needed to interpret the data correctly.

Store the result in a DataFrame called `counts`.

Your goal is to obtain:

* six correctly separated columns
* `intensidad` as an integer
* `ocupacion` and `vmed` as floats
* `fecha` as a datetime
* dates interpreted in **day/month/year** order
* accented characters displayed correctly

You will need some of the following `read_csv()` arguments:

`sep`, `decimal`, `thousands`, `encoding`, `parse_dates`, `dayfirst`

> Use what you observed in the raw file to decide the correct value for each argument.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(counts.dtypes)
counts.head()

In [ ]:
assert counts.shape == (21568, 6), f"Expected (21568, 6), got {counts.shape}"
assert counts["fecha"].dtype.kind == "M", "The fecha column is not a datetime"
assert counts["intensidad"].dtype.kind in "iu", "intensidad should be an integer"
assert counts["ocupacion"].dtype.kind == "f", "ocupacion should be a float"
assert counts["fecha"].min() == pd.Timestamp("2025-09-01"), (
    "The minimum date is not 1 September: check dayfirst")
assert counts["intensidad"].max() > 1000, (
    "If the maximum intensity is below 1000, the thousands point was misread")
print("Checks passed.")

> **Why `dayfirst` matters**
>
> A date such as `01/09/2025` is ambiguous: it could mean **1 September** or **9 January**, depending on the convention used.
>
> Setting `dayfirst=True` tells `pandas` how the dates in this file should be interpreted.
>
> The general rule is simple: **when you know the convention used by the source, state it explicitly rather than relying on automatic inference.**


In [ ]:
no_dayfirst = pd.read_csv(counts_path, sep=";", decimal=",", thousands=".",
                          encoding="latin-1", parse_dates=["fecha"])
print("dtype of fecha without dayfirst:", no_dayfirst["fecha"].dtype)
print("left as text?                 :", no_dayfirst["fecha"].dtype.kind != "M")
print()
print("The dangerous part comes now: min() and max() still work,")
print("but they compare strings, not dates.")
print("  min:", no_dayfirst["fecha"].min())
print("  max:", no_dayfirst["fecha"].max())
print()
print("Both look like the right answer. That is exactly the trap.")

### Think about it

If `fecha` remains as text instead of becoming a datetime, the problem may not appear immediately.

**At what later stage of an analysis could this cause an error or produce an incorrect result?**

Think, for example, about filtering, sorting, grouping or calculating time differences.

_Write your answer here:_


### Exercise 3. What does one row represent?

Before analysing a table, we need to understand its **granularity**: what does one row represent?

In this dataset, we expect each row to represent:

> **one hourly observation from one traffic measurement point**

If that is true, each combination of `id` and `fecha` should be unique.

Check this assumption.

Store:

- the number of fully duplicated rows in `n_duplicates`
- all rows involved in those duplicates in `dups`

Then check whether `(id, fecha)` is unique.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("Duplicated rows:", n_duplicates)
print("Unique (id, fecha) pairs:", not counts.duplicated(subset=["id", "fecha"]).any())
dups.head(4)

In [ ]:
assert n_duplicates == 40, f"Expected 40 duplicates, got {n_duplicates}"
assert len(dups) == 80, "dups should contain both copies of each duplicate"
print("Checks passed: the declared granularity does not hold.")

### Why are there duplicates?

These duplicate rows represent retransmissions from the field equipment, something that can occur in real traffic datasets.

For this exercise, we will treat them as duplicate observations.

Remove the duplicated rows, keeping the first occurrence.


In [ ]:
counts = counts.drop_duplicates().reset_index(drop=True)
assert len(counts) == 21528
print("Rows after removing duplicates:", len(counts))

### Exercise 4. Missing values and sentinels

According to the source documentation, a value of `-1` in `vmed` means:

> **no measurement available**

This is a **sentinel value**: a special value used to represent missing data.

If we leave `-1` as a normal number, it will be included in calculations such as the mean and will distort the result.

Your task:

1. Count how many `-1` values appear in `vmed` and store the result in `n_sentinel`.
2. Replace them with `np.nan`.
3. Compare the mean before and after the replacement.


In [ ]:
mean_before = counts["vmed"].mean()
# YOUR CODE HERE
raise NotImplementedError
mean_after = counts["vmed"].mean()
print(f"Sentinels: {n_sentinel}")
print(f"Mean before: {mean_before:.2f} km/h")
print(f"Mean after : {mean_after:.2f} km/h")
print(f"Bias introduced: {mean_after - mean_before:.2f} km/h")

In [ ]:
assert n_sentinel > 2000, "Expected more than 2000 sentinels"
assert counts["vmed"].isna().sum() == n_sentinel
assert (counts["vmed"].dropna() > 0).all(), "Some negative value remains in vmed"
assert mean_after - mean_before > 5, "The bias should be several km/h"
print("Checks passed.")

> **Why this matters**
>
> Many datasets use special values such as `-1`, `-9999` or `999` to represent missing observations.
>
> Unless these values are explicitly converted into missing values, Python will treat them as ordinary numbers and include them in calculations.
>
> The result may still look reasonable, which makes this type of error particularly easy to miss.


### Exercise 5. CSV versus Parquet

Now we will store the same table in two different formats and compare them.

**CSV** stores data as delimited text.  
**Parquet** is a binary, columnar format that also stores information about the schema.

We will compare three practical aspects:

- file size
- read time
- preservation of data types

Before running the experiment, make a prediction:

> **How much smaller do you expect the Parquet file to be: 2×, 10× or 50×?**

Save `counts` as `counts.parquet` in the working folder and complete the measurements.


In [ ]:
import time
parquet_path = WORK / "counts.parquet"
# YOUR CODE HERE
raise NotImplementedError

size_csv = counts_path.stat().st_size / 1e6
size_pq = parquet_path.stat().st_size / 1e6

t0 = time.perf_counter(); _ = pd.read_csv(counts_path, sep=";", decimal=",", thousands=".",
                                          encoding="latin-1", parse_dates=["fecha"], dayfirst=True)
t_csv = time.perf_counter() - t0
t0 = time.perf_counter(); reloaded = pd.read_parquet(parquet_path)
t_pq = time.perf_counter() - t0

print(f"CSV     {size_csv:6.2f} MB   {t_csv*1000:6.0f} ms")
print(f"Parquet {size_pq:6.2f} MB   {t_pq*1000:6.0f} ms")
print(f"Size ratio: {size_csv/size_pq:.1f}x")

In [ ]:
assert size_pq < size_csv, "The Parquet should be smaller"
assert reloaded["fecha"].dtype.kind == "M", (
    "After the round trip the date should still be a date")
assert reloaded.dtypes.equals(counts.dtypes), "Types were not preserved"
print("Checks passed: Parquet keeps the schema; the CSV loses it.")

### Reading only the columns we need

Columnar formats have another useful property: columns can be read selectively.

With Parquet, we can request only the columns needed for an analysis without loading the others.

CSV also allows column selection with `usecols`, but it remains a row-oriented text format that must be parsed from the source file.

Let's read only `id` and `intensidad` from the Parquet file.


In [ ]:
two_cols = pd.read_parquet(parquet_path, columns=["id", "intensidad"])
print(two_cols.shape, "->", list(two_cols.columns))

### What about small datasets?

Parquet is not always smaller than CSV.

Parquet files contain additional metadata and schema information, which introduces some fixed overhead. For very small tables, that overhead may be larger than the space saved through columnar storage and compression.

Let's test this with the locations table, which contains only 30 rows.


In [ ]:
loc_tmp = pd.read_csv(DATA / "pm_ubicaciones.csv", sep=";", decimal=",",
                      thousands=".", encoding="latin-1")
loc_tmp.to_parquet(WORK / "loc_tmp.parquet", index=False)
b_csv = (DATA / "pm_ubicaciones.csv").stat().st_size
b_pq = (WORK / "loc_tmp.parquet").stat().st_size
print(f"30 rows:  CSV {b_csv} bytes  |  Parquet {b_pq} bytes  ->  Parquet {b_pq / b_csv:.1f}x LARGER")
print("Columnar formats pay off at scale, not on thirty-row tables.")

### Think about it

Which column do you think contributes most to the reduction in file size?

Consider:

- repeated values
- numeric versus text data
- how many different values each column contains

_Write your hypothesis here:_


### Exercise 6. The question of the session

Our final question is:

> **At what elevation are the ten busiest traffic measurement points in Madrid?**

For this exercise, we will define **busiest** as the measurement points with the highest **mean hourly traffic intensity**.

In this lab, we will solve the **tabular part** of the problem:

1. calculate the mean intensity for each measurement point
2. select the ten highest values
3. join them with the locations table
4. save the result as `top10.parquet`

In **Lab 1.2**, we will use their coordinates to obtain the elevation.


In [ ]:
# YOUR CODE HERE
raise NotImplementedError
top10

In [ ]:
assert len(top10) == 10, f"Expected 10 rows, got {len(top10)}"
assert {"x_utm", "y_utm", "nombre"} <= set(top10.columns), (
    "A column from the locations table is missing: check the merge")
assert top10["x_utm"].notna().all(), "Some locations have no coordinate: the merge failed"
assert top10["mean_intensity"].is_monotonic_decreasing, "Sort from largest to smallest"
assert top10["x_utm"].between(432000, 452000).all(), "Coordinates outside the working area"
print("Checks passed.")

In [ ]:
top10.to_parquet(WORK / "top10.parquet", index=False)
print("Saved", WORK / "top10.parquet")
print("Lab 1.2 rebuilds it by itself if this session has expired, so nothing is lost.")

### Extension A. A typical public-sector Excel file

Excel is not the main focus of this lab, but it is still a common format for project data.

`imd_septiembre.xlsx` has a structure that you will often encounter in spreadsheets prepared for human readers rather than for direct analysis:

- a title and subtitle
- merged cells
- headers that do not start in the first row
- a separate sheet containing notes

First inspect the available sheets.

Then read the `Datos` sheet into a DataFrame called `imd`, skipping the introductory rows so that the final columns are:

`id`, `nombre`, `IMD (veh/día)`, `Ocupación (%)`, `V media (km/h)`


In [ ]:
sheets = pd.read_excel(DATA / "imd_septiembre.xlsx", sheet_name=None)
print("Sheets:", list(sheets))
sheets["Datos"].head(6)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
imd.head()

In [ ]:
assert list(imd.columns)[:2] == ["id", "nombre"], f"Wrong header: {list(imd.columns)}"
assert imd.shape == (30, 5), f"Expected (30, 5), got {imd.shape}"
assert imd["IMD (veh/día)"].dtype.kind in "iuf"
print("Checks passed.")

### Extension B. Hourly traffic profile

How does traffic intensity change throughout the day?

Plot the **mean traffic intensity by hour**, separating:

- weekdays
- weekends

Use the `.dt` accessor to extract the hour and day of the week from `fecha`.

What differences can you observe between the two profiles?


In [ ]:
import matplotlib.pyplot as plt
# YOUR CODE HERE
raise NotImplementedError


### Extension C. Data provenance

A dataset is more useful when we know **where it came from and how it was produced**.

Add three provenance fields to `top10`:

- `source_file`
- `extraction_date`
- `n_source_rows`

Save the result again as `top10_prov`.

In the next session, we will see why keeping this information alongside derived data becomes important.

> This follows the principle of documenting the origin and context of data described by Rule et al. (2019).


In [ ]:
# YOUR CODE HERE
raise NotImplementedError
top10_prov.head(2)

---

## Take-aways

By the end of this lab, you should be comfortable with five ideas:

- **Inspect before loading.** A file extension is useful information, but it does not guarantee the actual format or conventions used inside the file.
- **Reading a CSV requires assumptions.** Separators, decimal symbols, encodings and date formats are part of the data definition and should be made explicit.
- **Know what one row represents.** Granularity should be understood and checked, not simply assumed.
- **Missing data may be hidden.** Sentinel values must be identified and converted before analysis.
- **Storage format matters.** Parquet preserves data types, supports efficient column selection and can offer substantial storage and performance advantages for analytical datasets, particularly at larger scales.

The appropriate format depends on the purpose of the data: CSV remains useful for simple exchange and inspection, while Parquet is often better suited to repeated analytical workflows.